# Day 6 Baseline 建模：AI4I 2020 设备故障预测

本 notebook 的目标是完成第一版最小可行 baseline：在不做复杂特征工程、不做复杂调参的前提下，使用当前项目已经理解过的核心业务变量，训练一个可以解释、可以展示、可以衔接 Day 7 总结的故障预测模型。

当前阶段重点不是刷分，而是回答：仅使用基础运行状态字段时，模型能否识别少数故障样本。

## 1. Day 6 建模边界

- 目标变量：CSV 原始字段 `Machine failure`，对应 SQL 表中的 `Machine_failure`。
- 当前不使用 `TWF/HDF/PWF/OSF/RNF` 作为特征，因为它们是具体故障原因标记，更适合做解释或边界提醒；直接用于预测总故障标签容易造成标签泄漏。
- 当前不使用 `UDI` 和 `Product ID`，因为它们更像记录编号和产品编号，不是稳定的运行状态变量。
- 当前只做最小 baseline：Logistic Regression 和 XGBoost。
- 评价重点放在故障类 `1` 的 precision、recall、F1 和 confusion matrix，而不是只看 accuracy。

In [ ]:
# 导入基础数据处理、建模和评估库
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

from xgboost import XGBClassifier

# 统一随机种子，保证结果可复现
RANDOM_STATE = 42

## 2. 读取数据

当前 notebook 默认从项目根目录 `C:\2020 AI4I\ai4i_predictive_maintenance` 运行，因此直接读取 `data/raw/ai4i2020.csv`。

In [ ]:
df = pd.read_csv("data/raw/ai4i2020.csv")

print("数据规模：", df.shape)
df.head()

In [ ]:
# 快速确认字段和目标变量分布，避免建模前拿错列
print("字段列表：")
print(df.columns.tolist())

print("\n目标变量分布：")
print(df["Machine failure"].value_counts().sort_index())

print("\n目标变量占比：")
print((df["Machine failure"].value_counts(normalize=True).sort_index() * 100).round(2))

## 3. 选择特征与目标

特征选择沿用 Day 3 / Day 5 已经观察过的业务变量：产品类型、温度、转速、扭矩、刀具磨损。这里先不新增复杂组合特征，目的是建立第一版可比较的基础结果。

In [ ]:
target_col = "Machine failure"

feature_cols = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

leakage_cols = ["TWF", "HDF", "PWF", "OSF", "RNF"]
id_cols = ["UDI", "Product ID"]

print("本次使用的特征：", feature_cols)
print("本次暂不使用的编号字段：", id_cols)
print("本次暂不作为预测特征的故障原因字段：", leakage_cols)

X_raw = df[feature_cols].copy()
y = df[target_col].copy()

## 4. 基础预处理

`Type` 是类别字段，需要做 one-hot 编码。其余数值字段保持原始含义，不在第一版 baseline 中做复杂变换。

In [ ]:
# 对 Type 做最基础的 one-hot 编码；保留全部类别，方便后续解释
X = pd.get_dummies(X_raw, columns=["Type"], drop_first=False)

# XGBoost 不接受包含 [] 或 < 的特征名，这里只对建模矩阵列名做安全化处理
# 原始数据字段和业务含义不变，后续解释仍然可以对应回原始变量
safe_columns = (
    pd.Series(X.columns)
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("<", "lt", regex=False)
    .str.replace(">", "gt", regex=False)
    .str.replace(" ", "_", regex=False)
)
X.columns = safe_columns

print("建模特征列：")
print(X.columns.tolist())
print("\n建模矩阵规模：", X.shape)
X.head()

## 5. 划分训练集和测试集

故障样本占比只有约 3.39%，因此划分数据时使用 `stratify=y`，让训练集和测试集中的故障比例尽量保持一致。

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("训练集规模：", X_train.shape)
print("测试集规模：", X_test.shape)

print("\n训练集目标分布：")
print(y_train.value_counts().sort_index())

print("\n测试集目标分布：")
print(y_test.value_counts().sort_index())

## 6. 训练 baseline 模型

这里保留两个模型：

1. `Logistic Regression`：可解释性强，作为最基础的线性 baseline。
2. `XGBoost`：训练速度快，能捕捉非线性关系，作为当前 MVP 阶段的树模型 baseline。

由于故障样本明显少于正常样本，两个模型都使用最基础的不平衡处理参数，但不做复杂采样和调参。

In [ ]:
# Logistic Regression 对特征尺度敏感，因此只给它做标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
log_reg.fit(X_train_scaled, y_train)

# XGBoost 使用 scale_pos_weight 处理类别不平衡
negative_cnt = (y_train == 0).sum()
positive_cnt = (y_train == 1).sum()
scale_pos_weight = negative_cnt / positive_cnt

xgb_model = XGBClassifier(
    n_estimators=120,
    max_depth=3,
    learning_rate=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_model.fit(X_train, y_train)

print("模型训练完成。")
print("XGBoost scale_pos_weight：", round(scale_pos_weight, 2))

## 7. 模型评估

本项目的故障样本很少，因此不能只看 accuracy。这里重点看故障类 `1` 的 precision、recall 和 F1-score。

- precision：模型预测为故障的样本中，有多少是真的故障。
- recall：真实故障样本中，有多少被模型识别出来。
- F1-score：precision 和 recall 的综合表现。

In [ ]:
def evaluate_model(model_name, y_true, y_pred):
    """输出基础分类指标，并返回故障类的核心指标。"""
    print(f"===== {model_name} =====")
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("\nClassification report:")
    print(classification_report(y_true, y_pred, digits=3, zero_division=0))

    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        "model": model_name,
        "accuracy": report["accuracy"],
        "failure_precision": report["1"]["precision"],
        "failure_recall": report["1"]["recall"],
        "failure_f1": report["1"]["f1-score"],
    }


log_reg_pred = log_reg.predict(X_test_scaled)
xgb_pred = xgb_model.predict(X_test)

results = []
results.append(evaluate_model("Logistic Regression", y_test, log_reg_pred))
results.append(evaluate_model("XGBoost", y_test, xgb_pred))

In [ ]:
metrics_df = pd.DataFrame(results)
metrics_df = metrics_df.sort_values("failure_f1", ascending=False).reset_index(drop=True)
metrics_df

In [ ]:
# 用图表查看两个模型的混淆矩阵，方便 Day 7 做项目展示时引用
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, model_name, y_pred in zip(
    axes,
    ["Logistic Regression", "XGBoost"],
    [log_reg_pred, xgb_pred],
):
    cm = confusion_matrix(y_test, y_pred)
    ax.imshow(cm, cmap="Blues")
    ax.set_title(model_name)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="black")

plt.tight_layout()
plt.show()

## 8. 轻量查看 XGBoost 特征重要性

这里不是做复杂解释模型，只是查看第一版树模型更关注哪些基础变量，方便和 Day 3 / Day 5 的业务发现互相印证。

In [ ]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": xgb_model.feature_importances_,
}).sort_values("importance", ascending=False)

feature_importance

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(feature_importance["feature"], feature_importance["importance"], color="#4C78A8")
ax.invert_yaxis()
ax.set_title("XGBoost feature importance")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

## 9. Day 6 小结

本次 baseline 的目标是先跑通从业务分析到建模评估的最小闭环。当前模型只使用基础运行状态字段和产品类型，没有加入复杂特征工程，也没有使用具体故障原因字段 `TWF/HDF/PWF/OSF/RNF`，因此结果可以作为后续迭代的基础对照。

解读结果时需要优先关注故障类 `1` 的 recall 和 F1-score。如果模型 accuracy 很高但故障类 recall 很低，说明模型主要学会了识别多数类正常样本，对预测性维护场景的价值有限。

Day 7 可以基于本 notebook 的评估结果，整理 README、summary 和关键图表；如果后续继续迭代，再考虑加入少量业务特征，例如温差、功率近似值或高磨损标记。